# DATA-445: Exam 2 - Part 1

## Instructions

- Exam 2 - Part 1 is worth **100** points.
- Please submit a **.ipynb** file to Blackboard.
- **Please strive for clarity and organization.**
- **AI usage**: You may use AI exclusively for **debugging**. **Do not use AI** to generate answers.
- **Due Date: November 18, 2026, by 12:50 PM.**

## Exercise 1

Consider the `df_train.csv` datafile. This dataset contains airline passenger survey responses and their associated overall satisfaction outcome. The goal is to build classification models to predict `satisfaction`.

### (a) (2 points)
Read the datafile and create a data-frame called `df`. Remove any rows containing missing values.

In [7]:
import pandas as pd

df = pd.read_csv('df_train.csv')

df = df.dropna()

df.head()


,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,Female,Loyal Customer,67,Personal Travel,Eco,602,2,4,2,3,...,3,2,2,2,1,1,4,17,20.0,False
1,Male,Loyal Customer,36,Personal Travel,Eco,2475,3,2,3,3,...,3,1,1,4,3,3,3,122,184.0,False
2,Male,Loyal Customer,54,Business travel,Business,3658,3,1,1,1,...,4,3,3,3,3,1,2,43,55.0,False
3,Female,Loyal Customer,39,Business travel,Business,1846,1,1,1,1,...,4,5,5,4,5,5,4,3,20.0,True
4,Female,Loyal Customer,33,Business travel,Business,261,3,1,2,1,...,4,3,3,2,3,2,2,0,0.0,False


### (b) (4 points)
There are four object columns: `Gender`, `Customer Type`, `Type of Travel`, and `Class`. Convert those labels into numbers. *Hint*: you may consider using `OrdinalEncoder` from `sklearn.preprocessing`.

In [9]:
from sklearn.preprocessing import OrdinalEncoder

columns = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
df[columns] = OrdinalEncoder().fit_transform(df[columns])

df.head()

,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0.0,0.0,67,1.0,1.0,602,2,4,2,3,...,3,2,2,2,1,1,4,17,20.0,False
1,1.0,0.0,36,1.0,1.0,2475,3,2,3,3,...,3,1,1,4,3,3,3,122,184.0,False
2,1.0,0.0,54,0.0,0.0,3658,3,1,1,1,...,4,3,3,3,3,1,2,43,55.0,False
3,0.0,0.0,39,0.0,0.0,1846,1,1,1,1,...,4,5,5,4,5,5,4,3,20.0,True
4,0.0,0.0,33,0.0,0.0,261,3,1,2,1,...,4,3,3,2,3,2,2,0,0.0,False


### (c) (4 points)
Define `satisfaction` as the target variable, and the other variables as the input features. Also, define the 5-fold cross-validation strategy. Ensure to map `satisfaction` to 0-1 (`False` -> 0 and `True` -> 1)

In [10]:
from sklearn.model_selection import StratifiedKFold

df['satisfaction'] = df['satisfaction'].astype(int)

X = df.drop(columns=['satisfaction'])
y = df['satisfaction']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### (d) (15 points)
Train and evaluate the `RandomForestClassifier` model (`n_estimators=1000`, `max_depth=10`, and `random_state=42`) over the 5-fold cross-validation strategy defined in part (c). Use the `ROC-AUC` score as measure of performance.

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold 

rf_md = RandomForestClassifier(n_estimators=1000,
                               max_depth=10,
                               random_state=42,
                               n_jobs=-1)

rf_aucs = []
rf_accuracies = []
rf_recalls = []

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    rf_md.fit(X_train, y_train)
    y_prob = rf_md.predict_proba(X_test)[:, 1]
    y_pred = (y_prob > 0.1).astype(int)

    auc = roc_auc_score(y_test, y_prob)
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    
    rf_aucs.append(auc)
    rf_accuracies.append(accuracy)
    rf_recalls.append(recall)

print(f"Average ROC-AUC: {sum(rf_aucs)/len(rf_aucs):.4f}")
print(f"Average Accuracy: {sum(rf_accuracies)/len(rf_accuracies):.4f}")
print(f"Average Recall: {sum(rf_recalls)/len(rf_recalls):.4f}")
                            

Average ROC-AUC: 0.9894
Average Accuracy: 0.8593
Average Recall: 0.9914


### (e) (15 points)
Train and evaluate the `MLPClassifier` with two hidden layers: the first one with 8 neurons and `ReLU` as the activation function, the second one with 8 neurons and `ReLU` as the activation function. Also use `solver='adam'`, `max_iter=200` and `batch_size=256`, over the 5-fold cross validation strategy defined in part (c). Use the `ROC-AUC` score as measure of performance. Ensure to standardize the input features before you build the model.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

mlp_md = MLPClassifier(hidden_layer_sizes=(8, 8),
                       activation='relu',
                       solver='adam',
                       max_iter=200,
                       batch_size=256,
                       random_state=42)

mlp_aucs = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    mlp_md.fit(X_train_scaled, y_train)
    y_pred = mlp_md.predict_proba(X_test_scaled)[:, 1]
    
    score = roc_auc_score(y_test, y_pred)
    mlp_aucs.append(score)
    
    print(f"Fold {i+1} ROC-AUC: {score:.4f}")

print(f"\nAverage ROC-AUC: {sum(mlp_aucs)/len(mlp_aucs):.4f}")

### (f) (15 points)
Train and evaluate the `GradientBoostingClassifier` model (`n_estimators=1000`, `max_depth=8`, and `learning_rate=0.1`) over the 5-fold cross-validation strategy defined in part (c). Use the `ROC-AUC` score as measure of performance.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, recall_score

gb_md = GradientBoostingClassifier(n_estimators=1000,
                                   max_depth=8,
                                   random_state=42,
                                   learning_rate=0.01)
 
gb_aucs = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    gb_md.fit(X_train, y_train)
    y_pred_prob = gb_md.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_pred_prob)
    gb_aucs.append(score)
    
    print(f"Fold {i+1} ROC-AUC: {score:.4f}")

print(f"\nAverage ROC-AUC: {sum(gb_aucs)/len(gb_aucs):.4f}")

Fold 1 Accuracy: 0.9246995221315828, Recall: 0.9890012220864348
Fold 2 Accuracy: 0.9216102717574939, Recall: 0.9893357031770718


KeyboardInterrupt: 

### (g) (15 points)
Train and evaluate the `HistGradientBoostingClassifier` model (`max_iter=1000`, `max_depth=8`, and `learning_rate=0.1`) over the 5-fold cross-validation strategy defined in part (c). Use the `ROC-AUC` score as measure of performance.

In [ ]:

from sklearn.ensemble import HistGradientBoostingClassifier

hgb_md = HistGradientBoostingClassifier(max_iter=1000,
                                        max_depth=8,
                                        learning_rate=0.01,
                                        random_state=42)

hgb_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    hgb_md.fit(X_train, y_train)
    y_pred = hgb_md.predict_proba(X_test)[:, 1]
    score = recall_score(y_test, (y_pred > 0.1).astype(int))
    hgb_recalls.append(score)
    
    print(f"Fold {i+1} ROC-AUC: {score:.4f}")

print(f"\nAverage ROC-AUC: {sum(hgb_recalls)/len(hgb_recalls):.4f}")



### (h) (15 points)
Train and evaluate the `StackingClassifier`, using the models from parts 1(d), 1(e), 1(f), 1(g) as the base estimators, over the 5-fold cross-validation strategy defined in part (c). Use `LogisticRegression` as the `final_estimator`. Use the `ROC-AUC` score as measure of performance.

In [ ]:
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np


logistic_md = LogisticRegression(max_iter=1000,
                                 random_state=42)

stacking_md = StackingClassifier(
    estimators=[('rfc', rf_md),
                ('gb', gb_md),
                ('hb', hgb_md),
                ('mlpc', mlp_md)],
    final_estimator=logistic_md,
    cv=5,
    n_jobs=-1
)

stacking_aucs = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
 
    stacking_md.fit(X_train_scaled, y_train)
    
    y_pred_prob = stacking_md.predict_proba(X_test_scaled)[:, 1]
    
    score = roc_auc_score(y_test, y_pred_prob)
    stacking_aucs.append(score)
    print(f"Fold {i+1} ROC-AUC Score: {score:.4f}")

print("\nMean ROC-AUC Score:", np.mean(stacking_aucs))
    

### (i) (3 points)
Using the results from parts 1(d) to 1(h), what model would you use to predict `satisfaction`.

You would want something like HGB becuase the accuracy is a lot higher than others and it has high training speed.  

### (j) (12 points)
In this part you will explore **clustering as a feature engineering approach**. Use the input features defined in part (c) (do **not** include `satisfaction` when clustering).

1. **(4 points)** Standardize the input features and fit a `KMeans` model (`random_state=42`, `n_init=10`) for each number of clusters $k = 2, 3, \ldots, 10$. For each $k$, compute the **silhouette score**. *Hint*: since the dataset is large, you may use `silhouette_score` with `sample_size=10000` and `random_state=42`. Plot the silhouette score against $k$ and select the number of clusters, $k^*$, with the highest silhouette score.
2. **(5 points)** Using $k^*$, create a new categorical feature that contains the cluster membership of each observation, and one-hot encode it (e.g., `OneHotEncoder`). Build a `HistGradientBoostingClassifier` (same hyperparameters as in part 1(g)) using the original features plus the one-hot encoded cluster membership. Evaluate it over the same 5-fold cross-validation strategy defined in part (c), using the `ROC-AUC` score. To avoid data leakage, the scaler and `KMeans` must be fit **only on the training portion of each fold** (e.g., by using a `Pipeline`/`ColumnTransformer` with a custom transformer, or by looping over the folds manually).
3. **(3 points)** Compare the mean out-of-sample `ROC-AUC` of this model against the one obtained in part 1(g). Did adding the cluster membership feature improve performance? Briefly justify your answer.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

silhouette_scores = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_, sample_size=10000, random_state=42))

plt.plot(range(2, 11), silhouette_scores)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs. Number of Clusters')
plt.grid(True)
plt.show()

In [23]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

k = 3

hgb_aucs = []

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    train_clusters = kmeans.fit_predict(X_train_scaled).reshape(-1, 1)
    test_clusters = kmeans.predict(X_test_scaled).reshape(-1, 1)
    
    oh = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    train_oh = oh.fit_transform(train_clusters)
    test_oh = oh.transform(test_clusters)
    
    X_train = np.hstack([X_train.values, train_oh])
    X_test = np.hstack([X_test.values, test_oh])
    
    model = HistGradientBoostingClassifier(
        max_iter=1000, max_depth=8, learning_rate=0.01, random_state=42
    )
    model.fit(X_train, y_train)
    
    score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    hgb_aucs.append(score)
    print(f"Fold {i+1} ROC-AUC: {score:.4f}")

print(f"\nAverage ROC-AUC: {sum(hgb_aucs)/len(hgb_aucs):.4f}")

KeyboardInterrupt: 